In [2]:
%pip install -q -U langchain langchain-core langchain-openrouter langgraph python-dotenv pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
import json
import re
import time
import uuid

from pathlib import Path
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openrouter import ChatOpenRouter
from langgraph.checkpoint.memory import InMemorySaver

print("All ConductIQ dependencies imported successfully.")

All ConductIQ dependencies imported successfully.


In [15]:
# Locate the ConductIQ project directory
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "knowledge_base").exists():
    PROJECT_DIR = PROJECT_DIR.parent

KB_DIR = PROJECT_DIR / "knowledge_base"

# Verify project directory
if not KB_DIR.exists():
    raise FileNotFoundError(
        f"Knowledge base directory not found: {KB_DIR}"
    )

# Load conduct rules
with open(
    KB_DIR / "conduct_rules.json",
    "r",
    encoding="utf-8"
) as file:
    conduct_data = json.load(file)

conduct_rules = conduct_data["rules"]

# Load procedures and deadlines
with open(
    KB_DIR / "procedure_rules.json",
    "r",
    encoding="utf-8"
) as file:
    procedure_data = json.load(file)

procedure_rules = procedure_data["procedures"]
deadlines = procedure_data["deadlines"]

print("Knowledge bases loaded successfully.")
print("Conduct rules:", len(conduct_rules))
print("Procedures:", len(procedure_rules))
print("Deadlines:", len(deadlines))
print("Project directory:", PROJECT_DIR)

Knowledge bases loaded successfully.
Conduct rules: 29
Procedures: 14
Deadlines: 4
Project directory: c:\Users\user\Desktop\ConductIQ


In [17]:
# Define the conduct rule engine 
class ConductRuleEngine:

    STOPWORDS = {
        "the", "is", "are", "was", "were", "a", "an", "and",
        "or", "of", "to", "in", "on", "for", "with", "by",
        "under", "what", "how", "does", "do", "can", "could",
        "would", "should", "student", "university", "rub"
    }

    FIELD_WEIGHTS = {
        "rule_name": 4.0,
        "retrieval_keywords": 3.0,
        "category": 2.0,
        "description": 1.0,
        "source_quote": 0.5
    }

    def __init__(self, rules):
        self.rules = rules

    def normalize(self, text):
        text = str(text).lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def tokenize(self, text):
        return {
            word for word in self.normalize(text).split()
            if word not in self.STOPWORDS and len(word) > 1
        }

    def search(self, query, top_k=5):

        query_normalized = self.normalize(query)
        query_tokens = self.tokenize(query)

        if not query_tokens:
            return []

        results = []

        for rule in self.rules:

            score = 0.0
            matched_tokens = set()

            for field, weight in self.FIELD_WEIGHTS.items():

                value = rule.get(field, "")

                if isinstance(value, list):
                    field_text = " ".join(map(str, value))
                else:
                    field_text = str(value)

                field_tokens = self.tokenize(field_text)
                matches = query_tokens.intersection(field_tokens)

                score += len(matches) * weight
                matched_tokens.update(matches)

            coverage = len(matched_tokens) / len(query_tokens)

            rule_name = self.normalize(
                rule.get("rule_name", "")
            )

            if rule_name and rule_name in query_normalized:
                score += 5.0

            final_score = score * (0.5 + 0.5 * coverage)

            if final_score > 0:

                result = rule.copy()
                result["retrieval_score"] = round(final_score, 3)
                result["query_coverage"] = round(coverage, 3)

                results.append(result)

        results.sort(
            key=lambda x: x["retrieval_score"],
            reverse=True
        )

        return results[:top_k]


print("ConductRuleEngine defined successfully.")

ConductRuleEngine defined successfully.


In [40]:
# Define the procedureEngone 
class ProcedureEngine:

    STOPWORDS = {
        "the", "is", "are", "was", "were", "a", "an", "and",
        "or", "of", "to", "in", "on", "for", "with", "by",
        "under", "what", "how", "does", "do", "can", "could",
        "would", "should", "student", "university", "rub"
    }

    def __init__(self, procedures, deadlines):
        self.procedures = procedures
        self.deadlines = deadlines

    def normalize(self, text):
        text = str(text).lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def tokenize(self, text):
        return {
            word for word in self.normalize(text).split()
            if word not in self.STOPWORDS and len(word) > 1
        }

    def search_records(
        self,
        query,
        records,
        name_field,
        keyword_field,
        top_k=5
    ):

        query_normalized = self.normalize(query)
        query_tokens = self.tokenize(query)

        if not query_tokens:
            return []

        results = []

        for record in records:

            name = str(record.get(name_field, ""))

            description = str(record.get("description", ""))

            keywords = record.get(keyword_field, [])

            if isinstance(keywords, list):
                keyword_text = " ".join(map(str, keywords))
            else:
                keyword_text = str(keywords)

            name_tokens = self.tokenize(name)
            description_tokens = self.tokenize(description)
            keyword_tokens = self.tokenize(keyword_text)

            score = (
                len(query_tokens & name_tokens) * 4.0
                + len(query_tokens & keyword_tokens) * 3.0
                + len(query_tokens & description_tokens) * 0.5
            )

            matched_tokens = (
                name_tokens | keyword_tokens | description_tokens
            )

            coverage = (
                len(query_tokens & matched_tokens)
                / len(query_tokens)
            )

            if name and self.normalize(name) in query_normalized:
                score += 5.0

            final_score = score * (0.5 + 0.5 * coverage)

            if final_score > 0:

                result = record.copy()
                result["retrieval_score"] = round(final_score, 3)
                result["query_coverage"] = round(coverage, 3)

                results.append(result)

        results.sort(
            key=lambda x: x["retrieval_score"],
            reverse=True
        )

        return results[:top_k]

    def search_procedures(self, query, top_k=5):

        return self.search_records(
            query,
            self.procedures,
            "procedure_name",
            "procedure_keywords",
            top_k
        )

    def search_deadlines(self, query, top_k=4):
        name_field = "deadline_name"
        if self.deadlines and name_field not in self.deadlines[0]:
            name_field = "deadline_type"
        return self.search_records(
            query,
            self.deadlines,
            name_field,
            "deadline_keywords",
            top_k
        )

    def search(self, query, top_k=5):

        return {
            "procedures": self.search_procedures(query, top_k),
            "deadlines": self.search_deadlines(query, top_k)
        }


print("ProcedureEngine defined successfully.")

ProcedureEngine defined successfully.


In [41]:
# Evidence validation for ConductIQ


def extract_evidence_records(messages):
    """
    Extract evidence records returned by retrieval tools.
    """

    evidence_records = []

    for message in messages:

        if message.type != "tool":
            continue

        content = message.content

        if isinstance(content, list):
            content = "".join(
                str(item.get("text", item))
                if isinstance(item, dict)
                else str(item)
                for item in content
            )

        try:
            data = json.loads(content)

            # Normal retrieval tool output
            if isinstance(data, dict):

                if "records" in data:
                    evidence_records.extend(
                        data["records"]
                    )

                # Multi-step investigation output
                if "steps" in data:

                    for step in data["steps"]:

                        evidence = step.get(
                            "evidence", {}
                        )

                        evidence_records.extend(
                            evidence.get("records", [])
                        )

        except (json.JSONDecodeError, TypeError):
            continue

    return evidence_records


def validate_agent_evidence(answer, evidence_records):
    """
    Validate retrieved evidence and citations
    before returning the agent response.
    """

    issues = []

    if not evidence_records:
        return {
            "is_valid": False,
            "issues": ["No supporting evidence was retrieved."]
        }

    valid_records = []

    for record in evidence_records:

        section = record.get("section_number")
        page = record.get("source_page")

        if not section:
            issues.append(
                f"Missing section number in record: {record.get('id')}"
            )

        if not page:
            issues.append(
                f"Missing source page in record: {record.get('id')}"
            )

        if section and page:
            valid_records.append(record)

    if not valid_records:
        return {
            "is_valid": False,
            "issues": issues or [
                "No records contain complete citation metadata."
            ]
        }

    # Check whether the answer includes at least one
    # citation matching the retrieved evidence.
    citation_found = False

    for record in valid_records:

        section = str(record["section_number"])
        page = str(record["source_page"])

        section_present = (
            section.lower() in answer.lower()
        )

        page_pattern = (
            r"\b(?:printed\s+)?pages?\s*"
            + re.escape(page)
            + r"\b"
        )

        page_present = re.search(
            page_pattern,
            answer,
            re.IGNORECASE
        )

        if section_present and page_present:
            citation_found = True
            break

    if not citation_found:
        issues.append(
            "The answer does not contain a citation "
            "matching the retrieved evidence."
        )

    return {
        "is_valid": len(issues) == 0,
        "issues": issues
    }


print("ConductIQ evidence validation initialized.")

ConductIQ evidence validation initialized.


In [20]:
# Initialize the retrieval engines
conduct_engine = ConductRuleEngine(conduct_rules)

procedure_engine = ProcedureEngine(
    procedure_rules,
    deadlines
)

print("Both retrieval engines initialized successfully.")

Both retrieval engines initialized successfully.


In [30]:
# Load environment variables from the project root
load_dotenv(PROJECT_DIR / ".env")

# Retrieve API key
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY is missing. "
        "Please check your .env file."
    )

# Configure the LLM
llm = ChatOpenRouter(
    model=os.getenv(
        "OPENROUTER_MODEL",
        "openai/gpt-4.1-mini"
    ),
    temperature=0,
    max_tokens=1500,
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter LLM configured successfully.")

OpenRouter LLM configured successfully.


In [31]:
# Define evidence formatting
def format_evidence(records, record_type):
    """
    Format retrieved knowledge-base records
    into structured evidence for the LLM.
    """

    if not records:
        return {
            "record_type": record_type,
            "evidence_found": False,
            "records": []
        }

    formatted_records = []

    for record in records:

        formatted_records.append({
            "id": record.get(
                "rule_id",
                record.get(
                    "procedure_id",
                    record.get("deadline_id", "Not specified")
                )
            ),

            "name": record.get(
                "rule_name",
                record.get(
                    "procedure_name",
                    record.get(
                        "deadline_name",
                        record.get("deadline_type", "Not specified")
                    )
                )
            ),

            "section_number": record.get("section_number"),

            "source_page": record.get(
                "source_page",
                record.get("printed_page")
            ),
            "exceptions": record.get("exceptions", []),

            "description": record.get("description", ""),

            "source_quote": record.get("source_quote", ""),

            "conditions": record.get("conditions", []),

            "possible_measures": record.get(
                "possible_measures", []
            ),

            "retrieval_score": record.get("retrieval_score"),

            "query_coverage": record.get("query_coverage")
        })

    return {
        "record_type": record_type,
        "evidence_found": True,
        "records": formatted_records
    }


print("Evidence formatter initialized.")

Evidence formatter initialized.


In [32]:
# Define the ConductIQ retrival tools
@tool
def search_conduct_rules(query: str) -> str:
    """
    Search the RUB Student Code of Conduct 2026
    for misconduct categories and conduct rules.
    """

    results = conduct_engine.search(
        query,
        top_k=5
    )

    evidence = format_evidence(
        results,
        "conduct_rules"
    )

    return json.dumps(
        evidence,
        ensure_ascii=False,
        indent=2
    )


@tool
def search_procedures(query: str) -> str:
    """
    Search disciplinary procedures, reporting processes,
    student rights, and committee responsibilities.
    """

    results = procedure_engine.search_procedures(
        query,
        top_k=5
    )

    evidence = format_evidence(
        results,
        "procedures"
    )

    return json.dumps(
        evidence,
        ensure_ascii=False,
        indent=2
    )


@tool
def search_deadlines(query: str) -> str:
    """
    Search official working-day deadlines and
    appeal timeframes in the knowledge base.
    """

    results = procedure_engine.search_deadlines(
        query,
        top_k=4
    )

    evidence = format_evidence(
        results,
        "deadlines"
    )

    return json.dumps(
        evidence,
        ensure_ascii=False,
        indent=2
    )


@tool
def investigate_policy_question(query: str) -> str:
    """
    Perform a multi-step investigation by searching
    conduct rules, disciplinary procedures, and deadlines.

    Use this tool when a question requires information
    from multiple parts of the Student Code of Conduct.
    """

    conduct_results = conduct_engine.search(
        query,
        top_k=4
    )

    procedure_results = procedure_engine.search_procedures(
        query,
        top_k=4
    )

    deadline_results = procedure_engine.search_deadlines(
        query,
        top_k=4
    )

    investigation = {
        "investigation_type": "multi_step_policy_retrieval",

        "steps": [
            {
                "step": 1,
                "description": "Retrieve relevant conduct rules",
                "evidence": format_evidence(
                    conduct_results,
                    "conduct_rules"
                )
            },
            {
                "step": 2,
                "description": "Retrieve disciplinary procedures",
                "evidence": format_evidence(
                    procedure_results,
                    "procedures"
                )
            },
            {
                "step": 3,
                "description": "Retrieve relevant deadlines",
                "evidence": format_evidence(
                    deadline_results,
                    "deadlines"
                )
            }
        ]
    }

    return json.dumps(
        investigation,
        ensure_ascii=False,
        indent=2
    )


print("ConductIQ retrieval tools created.")

ConductIQ retrieval tools created.


In [33]:
# Register all tools
conductiq_tools = [
    search_conduct_rules,
    search_procedures,
    search_deadlines,
    investigate_policy_question
]

print("Registered ConductIQ tools:")

for tool_item in conductiq_tools:
    print("-", tool_item.name)

Registered ConductIQ tools:
- search_conduct_rules
- search_procedures
- search_deadlines
- investigate_policy_question


In [ ]:
# Create the imporved system prompt
CONDUCTIQ_SYSTEM_PROMPT = """
You are ConductIQ, an intelligent Student Code of Conduct
Assistant for the Royal University of Bhutan (RUB).

Your responsibility is to help students understand the
RUB Student Code of Conduct 2026 using the official
structured knowledge base.

You are an evidence-grounded, tool-using intelligent agent.

1. REASONING AND PLANNING

Before answering a question:

1. Understand the student's intention.
2. Identify the relevant policy topic.
3. Determine which evidence is required.
4. Select the appropriate retrieval tool or tools.
5. Retrieve relevant evidence.
6. Check whether the evidence supports the answer.
7. Generate a clear response with citations.

For questions involving multiple topics, use the
investigate_policy_question tool.

Do not answer policy questions based only on assumptions.

2. TOOL SELECTION

Use search_conduct_rules for:
- Misconduct categories
- Academic dishonesty
- Prohibited conduct
- Possible disciplinary measures

Use search_procedures for:
- Reporting misconduct
- Disciplinary processes
- Student rights
- Committee responsibilities
- Appeals procedures

Use search_deadlines for:
- Appeal deadlines
- Working-day timeframes
- Committee decision deadlines

Use investigate_policy_question for:
- Questions involving several policy areas
- Questions asking about both misconduct and procedures
- Questions requiring rules, procedures and deadlines together

You may call more than one specialized tool when needed.

3. EVIDENCE AND CITATIONS

Base policy-related answers on retrieved evidence.

Never invent:
- Rules
- Penalties
- Deadlines
- Exceptions
- Procedures
- Student rights

Every substantive policy claim must have a citation.

Use this format:

Source: RUB Student Code of Conduct 2026,
Section X.X.X, printed page XX.

Use the actual section and page metadata returned
by the knowledge base.

Do not invent a citation if the metadata is missing.

4. DISCIPLINARY PROCEDURES

Explain procedures in the correct sequence.

Distinguish between:
- Student Services Officer (SSO)
- Dean of Student Affairs (DSA)
- Conduct Disciplinary Committee (CDC)
- University Disciplinary Committee (UDC)

Do not convert working days into calendar days.

Explain appeal deadlines accurately.

5. STUDENT RIGHTS

Explain the student's right to:
- Know the allegations
- Respond to allegations
- Present supporting evidence

Only state rights supported by retrieved evidence.

6. DISCIPLINARY MEASURES

Explain possible disciplinary measures only when
supported by the official evidence.

Possible measures must not be presented as automatic
punishments or mandatory sequences.

Never determine whether a student is guilty.

Explain that disciplinary decisions must be
case-specific and proportionate where supported
by the Code.

7. CONVERSATIONAL MEMORY

Use previous messages in the current conversation
to understand follow-up questions.

Resolve references such as:
- It
- That misconduct
- The previous rule
- The appeal
- That procedure

Use conversational history to understand context.

However, previous conversation is not official evidence.

Retrieve the relevant policy evidence again before
answering substantive follow-up questions.

Do not mix information from separate conversations.

8. FEEDBACK AND SAFETY

Do not treat student feedback as an official
amendment to the Student Code of Conduct.

Do not modify official rules based on feedback.

If evidence is insufficient:
- Clearly explain the limitation.
- Do not guess.
- Recommend contacting the appropriate College
  authority or Student Services Officer.

9. RESPONSE STYLE

Use clear, professional, student-friendly language.

For simple questions, answer directly.

For complex questions, organize answers into:
- Relevant rule
- Applicable procedure
- Student rights
- Applicable deadline
- Official references

You are an informational assistant, not a
disciplinary authority.

Always prioritize the official RUB Student
Code of Conduct 2026.

ADDITIONAL EVIDENCE REQUIREMENTS

1. Use only information supported by retrieved knowledge-base records.

2. Do not assume that a general disciplinary procedure
   automatically applies to a specific misconduct category.

3. When a question concerns a specific misconduct category,
   distinguish between:
   - What the relevant misconduct rule explicitly states.
   - What general disciplinary procedures state.
   - What remains unspecified in the knowledge base.

4. For questions about appeals or timeframes, retrieve
   deadline evidence as well as procedure evidence.

5. Never use a citation from conversation history as
   a substitute for retrieving current evidence.

6. If evidence does not establish whether a procedure
   applies to a particular misconduct category, explain
   that limitation instead of making an assumption.
"""

In [35]:
# Add conversational memory to the agent
# Create the memory checkpointer
conductiq_memory = InMemorySaver()

print("ConductIQ conversational memory initialized.")

ConductIQ conversational memory initialized.


In [36]:
# Create the agent with memory 
conductiq_agent = create_agent(
    model=llm,
    tools=conductiq_tools,
    system_prompt=CONDUCTIQ_SYSTEM_PROMPT,
    checkpointer=conductiq_memory
)

print("ConductIQ intelligent agent created successfully.")

ConductIQ intelligent agent created successfully.


In [42]:
# Updated ConductIQ agent function

def ask_conductiq(
    question: str,
    thread_id: str = "conductiq_default"
):
    """
    Ask ConductIQ a question with conversational memory
    and evidence validation.
    """

    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    start_time = time.perf_counter()

    # Invoke the agent
    response = conductiq_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        },
        config=config
    )

    latency = time.perf_counter() - start_time

    messages = response.get("messages", [])

    # Extract final assistant response
    answer = ""

    for message in reversed(messages):

        if message.type == "ai" and message.content:

            if isinstance(message.content, str):
                answer = message.content

            else:
                answer = str(message.content)

            break

    # Extract tool calls
    tool_calls = []

    for message in messages:

        if message.type == "ai":

            for tool_call in getattr(
                message,
                "tool_calls",
                []
            ):

                tool_calls.append(
                    tool_call.get("name")
                )

    # Extract evidence from tool responses
    evidence_records = extract_evidence_records(
        messages
    )

    # Validate evidence and citations
    validation = validate_agent_evidence(
        answer,
        evidence_records
    )

    # Abstain when evidence or citation validation fails
    if not validation["is_valid"]:

        answer = (
            "I could not verify sufficient supporting "
            "evidence and citations for this answer in "
            "the available knowledge base.\n\n"
            "Please contact your Student Services Officer "
            "or the appropriate College authority for "
            "clarification."
        )

    return {
        "question": question,
        "answer": answer,
        "thread_id": thread_id,
        "tool_calls": tool_calls,
        "evidence_count": len(evidence_records),
        "validation_passed": validation["is_valid"],
        "validation_issues": validation["issues"],
        "latency_seconds": round(latency, 3)
    }


print("ConductIQ evidence-validated agent function initialized.")

ConductIQ evidence-validated agent function initialized.


In [43]:
# Test conversational memory
# Create a conversation
test_thread_id = str(uuid.uuid4())

questions = [
    "What is academic dishonesty under the RUB Student Code of Conduct 2026?",

    "What disciplinary procedure applies to it?",

    "Who is responsible for handling the case?",

    "What if the student wants to appeal?"
]

for question in questions:

    result = ask_conductiq(
        question,
        thread_id=test_thread_id
    )

    print("=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    print("\nANSWER:\n")
    print(result["answer"])

    print("\nTOOLS USED:")
    print(result["tool_calls"])

    print("\nLATENCY:")
    print(result["latency_seconds"], "seconds")

QUESTION: What is academic dishonesty under the RUB Student Code of Conduct 2026?

ANSWER:

Academic dishonesty under the RUB Student Code of Conduct 2026 is defined and dealt with according to the provisions in the Wheel of Academic Law and other applicable University policies and regulations. The Code itself refers students to these specific documents for the detailed definition and handling of academic dishonesty.

Source: RUB Student Code of Conduct 2026, Section 3.1.1, page 9.

TOOLS USED:
['search_conduct_rules']

LATENCY:
5.737 seconds
QUESTION: What disciplinary procedure applies to it?

ANSWER:

The disciplinary procedure for academic dishonesty at RUB is governed by the provisions of the Wheel of Academic Law. The Student Code of Conduct 2026 specifies that cases of academic dishonesty shall be dealt with according to that document.

In general, disciplinary procedures must be fair, consistent, prompt, and impartial, respecting the rights and confidentiality of all involved. 